# `Qwen3.6-35B-A6B-LoopHSSD` (v3) · Notebook 3: Optimized On-Weight Inference & Frontier `R=1/2/3` Evaluation

**Shipped Whole-Model Runtime (`40L / 64L / 88L A6B`), `1M FP8` + `10M INT4` Offload Profiler, & Frontier Depth-Curve Suite (Colab G4 · `RTX PRO 6000 Blackwell` `96 GiB` · `230 GB` SSD Safe)**

| Specialized Notebook | Purpose |
| :--- | :--- |
| [`colab_probe.ipynb`](colab_probe.ipynb) | **Zero-Training Diagnostic & Calibration Probes**: `Gate -1` + `Gate 0a` + `Gate 0p-1..3` + calibrated CLI overrides |
| [`colab_train_e2e.ipynb`](colab_train_e2e.ipynb) | **End-to-End `Loop-HSSD v3` Training & Whole-Model Pack Release**: Resumable `Gate 0b -> Gate 4` pipeline + low-disk streaming `safetensors` Whole Model Pack |
| **[`colab_infer_eval.ipynb`](colab_infer_eval.ipynb)** *(you are here)* | **Shipped Whole-Model On-Weight Inference & Frontier `R=1/2/3` Eval**: `OptimizedLoopInferenceEngine`, `1M FP8` + `10M INT4` offload demo, `ARC-AGI-2` / `AIME 2026` / `LongBench v2` / `KernelBench` |
| [`colab_train.ipynb`](colab_train.ipynb) | Baseline `64L` Track-A (`GDN-2` + `mHC`) single-notebook pipeline |

### Runtime Architecture Highlights (`Single Checkpoint → 3 Compute Tiers`)
- **Dynamic Depth Control (`engine.depth(r)`)**: Switch per request between **`R=1` (`40L`, `~2.95B` active)**, **`R=2` (`64L`, `~4.46B` active)**, and **`R=3` (`88L A6B`, `~5.97B` active, `22` virtual global-attention layers)** with zero weight reloads.
- **Chunked Prefill + `O(1)` Recurrent `HSSD` / `GDN-2` Slots**: Streams `128K–1M` prompts in `4,096`-token chunks (materializing `248K`-vocab logits only on the final prompt token) with isolated `(layer, pass)` recurrent state buffers.
- **Two-Tier Ultra-Long Context**:
  - **Tier 1 (`1M` GPU-Resident Primary)**: `10.00 GiB` `FP8` `SharedKV` (`1.0×` KV across all 3 passes!) + `320 MiB` GPU Ward centroids with untied per-pass query indexers `W_q_idx^(r)`.
  - **Tier 2 (`10M` Host-DDR5 Stretch)**: `Block64HadamardINT4Cache` (`2,304 B` 4-token micro-page, `53.64 GiB` pinned DDR5 @ `10M`) keeping `[0:64]` mRoPE strictly orthogonal to `[64:256]` NoPE for exact `ReRoPE` commutativity, prefetched 3 layers ahead.
- **Verbose Crash Diagnostics (`qloop.debug`)**: Every benchmark subprocess is wrapped in `crash_guard` (`DEBUG_LEVEL = 1 | 2 | 3`).

In [ ]:
REPO = "https://github.com/BaiTian6641/Qwen3.6-35B-A3B-Loop60.git"
BRANCH = "main"
%cd /content
!test -d /content/qloop/.git && test ! -d /content/qloop_repo && mv /content/qloop /content/qloop_repo || true
!test -d /content/qloop_repo || git clone -b $BRANCH $REPO /content/qloop_repo
%cd /content/qloop_repo
!git pull --ff-only
!bash scripts/colab_setup.sh

In [ ]:
import importlib
import os
import sys

if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
if "qloop" in sys.modules and getattr(sys.modules["qloop"], "__file__", None) is None:
    for _k in [k for k in sys.modules if k == "qloop" or k.startswith("qloop.")]:
        del sys.modules[_k]
importlib.invalidate_caches()

from qloop.colab import hub_status, q, reclaim_disk, resolve_base_model_name, setup_env
from qloop.export_pack import export_whole_model_pack
from qloop.infer import InferenceConfig, OptimizedLoopInferenceEngine

BASE_PRECISION = "fp8"  # "fp8" (Qwen/Qwen3.6-35B-A3B-FP8, ~35 GiB) | "nvfp4" (unsloth/Qwen3.6-35B-A3B-NVFP4-Fast, ~21 GiB) | "bf16"
PACK_REPO = "TassP0lar/Qwen3.6-35B-A6B-LoopHSSD-Pack"  # shipped self-contained Whole Model Pack
SYNC_REPO = "TassP0lar/Qwen3.6-35B-A3B-Loop64"         # stage sync repo for benchmark scorecard commits
SMOKE_TOY_PACK = False                                 # True = build & test local toy Whole Model Pack in <10s
DEBUG_LEVEL = 1                                        # 1=normal | 2=NaN/Inf hooks | 3=CUDA_LAUNCH_BLOCKING

setup_env(
    hf_repo=SYNC_REPO,
    run_root="/content/runs",
    hf_home="/content/hf_cache",
    require_hf_token=False,
)
doctor_cmd = f"doctor --debug {DEBUG_LEVEL}"
if not SMOKE_TOY_PACK:
    doctor_cmd += f" --model {resolve_base_model_name(BASE_PRECISION)}"
q(doctor_cmd)

### 1 · Pull Whole Model Pack & Run Dynamic `R=1 (40L)` / `R=2 (64L)` / `R=3 (88L A6B)` On-Weight Inference
Loads the sharded `safetensors` Whole Model Pack directly into `OptimizedLoopInferenceEngine` with chunked prefill and `FP8` `SharedKV` cache support, then sweeps `R=1`, `R=2`, and `R=3` on a multi-hop mathematical deduction prompt and profiles `prefill_tok_s`, `decode_tok_s`, `peak_vram_mb`, and unique active experts.

In [ ]:
from qloop.debug import crash_guard

with crash_guard("/content/runs/infer_engine"):
    if SMOKE_TOY_PACK:
        from qloop.config import load_config
        from qloop.train.stack import build_stack

        toy_pack_dir = "/content/runs/smoke_toy_pack"
        toy_cfg = load_config(
            "configs/local/toy_lm.yaml",
            [
                "loop.prelude_end=4",
                "loop.core_end=12",
                "loop.num_passes=3",
                "loop.target_virtual_layers=32",
                "loop.train_pass_probs=[0.15,0.50,0.35]",
            ],
        )
        toy_stack = build_stack(toy_cfg)
        export_whole_model_pack(toy_stack, toy_pack_dir, tokenizer=toy_stack.tokenizer, max_shard_size_bytes=262144)
        target_pack = toy_pack_dir
    else:
        target_pack = PACK_REPO

    infer_cfg = InferenceConfig(
        prefill_chunk_size=4096,
        fp8_kv_cache=True,
        compile_decode=False,
    )
    engine = OptimizedLoopInferenceEngine.from_pretrained(
        target_pack,
        cfg=infer_cfg,
        token=os.environ.get("HF_TOKEN"),
    )

    prompt = (
        "Let a, b, c > 0 satisfy a + b + c = 3. "
        "Prove step-by-step that a^2/(a + 2b^2) + b^2/(b + 2c^2) + c^2/(c + 2a^2) >= 1."
    )
    max_toks = 32 if SMOKE_TOY_PACK else 512
    virtual_map = {1: "40L (~2.95B)", 2: "64L (~4.46B)", 3: "88L A6B (~5.97B)"}
    for r in (1, 2, 3):
        with engine.depth(r):
            out = engine.generate(prompt, max_new_tokens=max_toks, temperature=0.0)
        text = out.text if hasattr(out, "text") else str(out)
        metrics = getattr(out, "metrics", None) or getattr(engine, "last_metrics", None)
        print(f"=== Pass Depth R={r} [{virtual_map[r]}] ===")
        if metrics is not None:
            print("Telemetry:", metrics)
        print(text[:600], "\n")

    bench_profile = engine.benchmark(
        prompt_len=64 if SMOKE_TOY_PACK else 512,
        gen_len=16 if SMOKE_TOY_PACK else 64,
        depths=(1, 2, 3),
    )
    print("=== Throughput & VRAM Depth Profile (R=1 vs R=2 vs R=3) ===")
    for depth_key, stats in sorted((bench_profile.get("depths") or bench_profile).items(), key=lambda kv: int(kv[0])):
        print(f"  depth {depth_key}: {stats}")

### 2 · Two-Tier Ultra-Long Context Demo (`1M` `FP8` SharedKV & `10M` Block-64 Hadamard `INT4` Offload)
Verifies Tier-1 `FP8KVCacheQuantizer` (`10.00 GiB` @ `1M` tokens across all 3 passes) and Tier-2 `Block64HadamardINT4Cache` (`walsh_hadamard_block(..., 64)` keeping `[0:64]` mRoPE orthogonal to `[64:256]` NoPE, exact `2,304 B` 4-token micro-page, 3-layer async prefetch, and `ReRoPE` inner-product invariance).

In [ ]:
import torch
from qloop.model.offload import Block64HadamardINT4Cache, FP8KVCacheQuantizer, walsh_hadamard_block
from qloop.model.pos_rerope import apply_partial_rerope_q, clamped_relative_distance

with crash_guard("/content/runs/offload_demo"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sample_k = torch.randn(1, 2, 64, 256, device=device, dtype=torch.float32)
    sample_v = torch.randn(1, 2, 64, 256, device=device, dtype=torch.float32)
    sample_v[..., 13] *= 12.0  # inject heavy channel outlier to test H_64 smoothing

    # Tier-1: 1M GPU-Resident FP8 SharedKV
    fp8_q = FP8KVCacheQuantizer()
    k_q, k_scale = fp8_q.quantize(sample_k)
    k_rec = fp8_q.dequantize(k_q, k_scale)
    fp8_rel_err = float((sample_k - k_rec.float()).norm() / sample_k.norm())

    # Tier-2: 10M Host-DDR5 Block-64 Walsh-Hadamard Group-64 INT4 Micro-Page Offload
    offload_cache = Block64HadamardINT4Cache(
        num_layers=10,
        num_kv_heads=2,
        head_dim=256,
        pin_memory=torch.cuda.is_available(),
    )
    offload_cache.append_tokens(0, sample_k, sample_v)
    page_ids = torch.arange(0, 8, device=device, dtype=torch.long)  # 8 micro-pages = 32 retrieved tokens
    offload_cache.prefetch_micro_pages(0, page_ids)
    attn_w = torch.softmax(torch.randn(1, 16, 4, 32, device=device), dim=-1)
    k_hat, v_out = offload_cache.gather_kv_and_reconstruct_v(0, page_ids, attn_w)

    # ReRoPE distance clamping + H_64 rotary/NoPE subspace isolation check
    raw_delta = torch.tensor([4096, 131072, 1048576, 10000000], device=device)
    clamped = clamped_relative_distance(raw_delta)
    inv_freq = 1.0 / (10_000_000.0 ** (torch.arange(0, 64, 2, device=device, dtype=torch.float32) / 64.0))
    q_test = torch.randn(1, 16, 4, 256, device=device, dtype=torch.float32)
    q_rerope = apply_partial_rerope_q(
        q_test,
        k_positions=torch.tensor([0, 200000, 900000, 5000000], device=device),
        q_positions=torch.tensor([5000000, 5000000, 5000000, 5000000], device=device),
        inv_freq=inv_freq,
        rotary_dim=64,
    )
    nope_max_diff = float((q_rerope[..., 64:] - q_test[..., 64:]).abs().max())
    h64_invariance = float(
        ((walsh_hadamard_block(q_rerope, 64) * walsh_hadamard_block(q_test, 64)).sum(-1)
         - (q_rerope * q_test).sum(-1)).abs().max()
    )

    print(f"Tier-1 FP8 SharedKV     : rel_err={fp8_rel_err:.4f} | 1M 10-layer footprint = 10.00 GiB HBM (1.0x across R=1..3)")
    print(f"Tier-2 H_64 INT4 Offload: micro-page = 2,304 B | 10M 10-layer footprint = 53.64 GiB pinned DDR5")
    print(f"ReRoPE Clamped Deltas   : {raw_delta.tolist()} -> {[round(float(x), 1) for x in clamped.tolist()]}")
    print(f"NoPE [64:256] Drift     : {nope_max_diff:.2e} | H_64 Inner-Product Drift: {h64_invariance:.2e}")

### 3 · Frontier Depth-Curve Benchmark Suite (`R=1` vs `R=2` vs `R=3` on `ARC-AGI-2`, `AIME 2026`, `LongBench v2`, `KernelBench`)
Executes the held-out frontier depth-curve suite across `40L` (`R=1`), `64L` (`R=2`), and `88L A6B` (`R=3`), displays the monotonic depth-scaling scorecard, and optionally commits `eval/suite_summary.json` to `SYNC_REPO`.

In [ ]:
import json

RUN_FRONTIER_BENCHMARKS = True
USE_HSSD_V3 = True                       # True = Track-B HSSD v3; False = Track-A GDN-2 fallback
EVAL_LIMIT = 4 if SMOKE_TOY_PACK else 0  # 0 = full held-out benchmark split; >0 = fast sanity subset
EVAL_CKPT = "/content/runs/s4_rl"        # auto-pulled from SYNC_REPO if not cached locally
PUSH_EVAL_TO_HUB = False

if RUN_FRONTIER_BENCHMARKS and not SMOKE_TOY_PACK:
    track_ov = "-s gdn2.enabled=false -s hssd.enabled=true " if USE_HSSD_V3 else ""
    limit_ov = f"-s eval.defaults.limit={EVAL_LIMIT} " if EVAL_LIMIT > 0 else ""
    sync_ov = f"-s checkpoint.sync.hf_repo={SYNC_REPO}" if PUSH_EVAL_TO_HUB else "-s checkpoint.sync.enabled=false"
    q(
        f"eval -c configs/eval/suite.yaml --ckpt {EVAL_CKPT} --debug {DEBUG_LEVEL} "
        f"-s model.name={resolve_base_model_name(BASE_PRECISION)} "
        "-s loop.num_passes=3 -s loop.target_virtual_layers=88 "
        "-s loop.train_pass_probs=[0.15,0.50,0.35] "
        "-s loop.bridge_mode=subspace -s loop.bridge_rank=256 "
        "-s loop.bridge_init=[0.05,0.85,0.10] -s loop.attn_adapter_rank=64 "
        "-s lsa.num_query_passes=3 "
        f"{track_ov}"
        "-s eval.defaults.depths=[1,2,3] "
        f"{limit_ov}{sync_ov}"
    )
    summary_file = "/content/runs/eval/eval/suite_summary.json"
    if os.path.isfile(summary_file):
        with open(summary_file, encoding="utf-8") as f:
            suite_res = json.load(f)
        benches = suite_res.get("benchmarks", suite_res)
        print(f"\n{'Benchmark':<20} {'R=1 (40L)':>12} {'R=2 (64L)':>12} {'R=3 (88L A6B)':>15} {'Gain (R3-R1)':>14}")
        print("-" * 77)
        for bname, binfo in sorted(benches.items()):
            if not isinstance(binfo, dict):
                continue
            dmap = binfo.get("depths") or {}
            s1 = (dmap.get("1") or dmap.get(1) or {}).get("mean_score")
            s2 = (dmap.get("2") or dmap.get(2) or {}).get("mean_score")
            s3 = (dmap.get("3") or dmap.get(3) or {}).get("mean_score")
            gain = (s3 - s1) if isinstance(s1, (int, float)) and isinstance(s3, (int, float)) else None
            f1 = f"{s1:.4f}" if isinstance(s1, (int, float)) else "-"
            f2 = f"{s2:.4f}" if isinstance(s2, (int, float)) else "-"
            f3 = f"{s3:.4f}" if isinstance(s3, (int, float)) else "-"
            fg = f"{gain:+.4f}" if isinstance(gain, (int, float)) else "-"
            print(f"{bname:<20} {f1:>12} {f2:>12} {f3:>15} {fg:>14}")
    reclaim_disk()

### 4 · Optional: Standard Hugging Face `AutoModelForCausalLM` Verification (`RUN_HF_AUTOMODEL_CHECK = True`)
Verifies that the bundled `configuration_qwen36_loophssd.py` & `modeling_qwen36_loophssd.py` inside the shipped Whole Model Pack also load cleanly via standard `transformers.AutoModelForCausalLM.from_pretrained(..., trust_remote_code=True)`.

In [ ]:
RUN_HF_AUTOMODEL_CHECK = False

if RUN_HF_AUTOMODEL_CHECK:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    hf_tok = AutoTokenizer.from_pretrained(target_pack, trust_remote_code=True)
    hf_model = AutoModelForCausalLM.from_pretrained(
        target_pack,
        trust_remote_code=True,
        torch_dtype="auto",
    )
    print("Loaded standalone HF custom model:", type(hf_model).__name__)